# GFIP Phase 2 — Exploratory Data Analysis

**Master Panel:** `data/processed/master_panel.parquet`  
**Coverage:** 17,070 rows × 35 columns, 274 countries, 1946–2025

## Workplan
1. Load and profile the Master Panel
2. Global distributions of all continuous variables
3. Temporal trends — global medians by decade
4. Bivariate correlations — H1–H7 exposure–outcome scatter plots
5. Missingness analysis — heatmap by variable × decade
6. Outlier investigation — flag extreme values
7. Key EDA findings for Phase 3

**Note:** This notebook is not TDD'd — EDA is exploratory by design (see TDD contract §Where strict TDD is not applicable).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

DATA = Path('..') / 'data'
panel = pd.read_parquet(DATA / 'processed' / 'master_panel.parquet')

# Conflict indicators: NaN means no conflict (0), not missing
panel['ucdp_conflict_binary'] = panel['ucdp_conflict_binary'].fillna(0).astype(int)
panel['ucdp_conflict_count'] = panel['ucdp_conflict_count'].fillna(0).astype(int)

# Analysis window: 1990+ where most outcome data exists
recent = panel[panel['year'] >= 1990].copy()

print(f'Full panel:   {len(panel):,} rows | {panel.iso3.nunique()} countries | {panel.year.min()}–{panel.year.max()}')
print(f'1990+ panel:  {len(recent):,} rows | {recent.iso3.nunique()} countries')

## 1. Variable Coverage (1990+)

In [ ]:
OUTCOME_VARS = [
    'gdp_pc_ppp', 'agri_value_added_pct_gdp', 'gini',
    'fsi_score', 'fsi_p1_legitimacy',
    'ucdp_conflict_binary', 'homicide_rate',
    'life_expectancy', 'u5mr',
    'refugee_outflow', 'idp_count',
    'renewable_freshwater_percap', 'total_withdrawal_km3', 'agri_withdrawal_pct',
    'safe_water_access_pct', 'population',
]

cov = recent[OUTCOME_VARS].notna().mean().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
cov.plot.barh(ax=ax, color=['#d62728' if v < 0.5 else '#2ca02c' if v > 0.8 else '#ff7f0e' for v in cov])
ax.axvline(0.6, color='red', linestyle='--', alpha=0.5, label='60% threshold')
ax.set_xlabel('Coverage (1990+ rows)')
ax.set_title('Variable coverage — 1990 to 2025')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Global Distributions

In [ ]:
DIST_VARS = [
    ('renewable_freshwater_percap', True, 'Freshwater per capita (m3/person/yr)'),
    ('gdp_pc_ppp', True, 'GDP per capita (const 2015 USD)'),
    ('life_expectancy', False, 'Life expectancy (years)'),
    ('u5mr', False, 'Under-5 mortality (per 1000 births)'),
    ('fsi_score', False, 'FSI score (higher = more fragile)'),
    ('homicide_rate', True, 'Homicide rate (per 100k)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, log, label) in zip(axes.flat, DIST_VARS):
    data = recent[col].dropna()
    if log:
        data = np.log10(data.clip(lower=0.1))
        label = f'log10({label})'
    sns.histplot(data, ax=ax, bins=40, color='steelblue', alpha=0.7)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'{col}\nn={len(data):,}')
plt.suptitle('Global distributions (1990+, log scale where skewed)', y=1.01)
plt.tight_layout()
plt.show()

## 3. Temporal Trends — Global Medians by Decade

In [ ]:
panel['decade'] = (panel['year'] // 10) * 10

TREND_VARS = [
    ('renewable_freshwater_percap', 'Freshwater per capita (m3)'),
    ('gdp_pc_ppp', 'GDP per capita (USD)'),
    ('life_expectancy', 'Life expectancy'),
    ('u5mr', 'Under-5 mortality rate'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (col, label) in zip(axes, TREND_VARS):
    trend = panel.groupby('year')[col].median().dropna()
    ax.plot(trend.index, trend.values, color='steelblue', linewidth=2)
    ax.set_xlabel('Year')
    ax.set_ylabel(label)
    ax.set_title(label)
plt.suptitle('Global median trends over time', y=1.02)
plt.tight_layout()
plt.show()

## 4. Bivariate Correlations — H1 through H7

Primary exposure: `renewable_freshwater_percap` (log scale due to right skew)

In [ ]:
# Log-transform the skewed exposure variable
recent['log_freshwater'] = np.log10(recent['renewable_freshwater_percap'].clip(lower=1))

H_PAIRS = [
    ('log_freshwater', 'gdp_pc_ppp', 'H1: Freshwater -> GDP per capita', True),
    ('log_freshwater', 'fsi_score', 'H2: Freshwater -> State fragility', False),
    ('log_freshwater', 'ucdp_conflict_binary', 'H3: Freshwater -> Conflict (binary)', False),
    ('log_freshwater', 'life_expectancy', 'H4: Freshwater -> Life expectancy', False),
    ('log_freshwater', 'refugee_outflow', 'H5: Freshwater -> Refugee outflow', True),
    ('log_freshwater', 'gini', 'H6: Freshwater -> Inequality (Gini)', False),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (x, y, title, log_y) in zip(axes.flat, H_PAIRS):
    df = recent[[x, y, 'iso3']].dropna()
    y_vals = np.log10(df[y].clip(lower=1)) if log_y else df[y]
    ax.scatter(df[x], y_vals, alpha=0.15, s=8, color='steelblue')
    # Add trend line
    if len(df) > 10:
        z = np.polyfit(df[x], y_vals, 1)
        p = np.poly1d(z)
        xr = np.linspace(df[x].min(), df[x].max(), 100)
        ax.plot(xr, p(xr), 'r-', linewidth=1.5, alpha=0.8)
        corr = df[x].corr(y_vals)
        ax.set_title(f'{title}\nn={len(df):,}, r={corr:.2f}', fontsize=9)
    ax.set_xlabel('log10(freshwater per capita)')
    ax.set_ylabel(f'log10({y})' if log_y else y)
plt.suptitle('H1-H6: Freshwater vs. human outcomes (1990+)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix — All Key Variables

In [ ]:
CORR_VARS = [
    'log_freshwater', 'gdp_pc_ppp', 'life_expectancy', 'u5mr',
    'fsi_score', 'ucdp_conflict_binary', 'homicide_rate',
    'refugee_outflow', 'gini', 'safe_water_access_pct',
]

corr_df = recent[CORR_VARS].copy()
corr_df['gdp_pc_ppp'] = np.log10(corr_df['gdp_pc_ppp'].clip(lower=1))
corr_df['refugee_outflow'] = np.log10(corr_df['refugee_outflow'].clip(lower=1))
corr_df['homicide_rate'] = np.log10(corr_df['homicide_rate'].clip(lower=0.1))

corr_matrix = corr_df.corr(method='spearman')  # Spearman: robust to non-linearity

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    ax=ax, square=True, linewidths=0.5
)
ax.set_title('Spearman correlations — all key variables (1990+)')
plt.tight_layout()
plt.show()

print('Key freshwater correlations:')
fw_corr = corr_matrix['log_freshwater'].drop('log_freshwater').sort_values()
for var, r in fw_corr.items():
    print(f'  {var:<30} r = {r:+.3f}')

## 6. Missingness Analysis

In [ ]:
# Missingness by variable and decade
panel['decade'] = (panel['year'] // 10) * 10
miss_by_decade = (
    panel[panel['year'] >= 1960]
    .groupby('decade')[OUTCOME_VARS]
    .apply(lambda g: g.notna().mean())
)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    miss_by_decade.T, annot=True, fmt='.0%',
    cmap='YlGn', vmin=0, vmax=1, ax=ax,
    linewidths=0.5
)
ax.set_title('Variable coverage by decade (% non-null)')
ax.set_xlabel('Decade')
plt.tight_layout()
plt.show()

## 7. Top Water-Stressed Countries (1990–2024)

In [ ]:
# Countries with most severe freshwater decline
fw_trend = (
    recent[recent['year'] >= 2000]
    .groupby('iso3')['renewable_freshwater_percap']
    .apply(lambda x: x.dropna())
)

# Countries with lowest absolute freshwater per capita (2015-2024 average)
recent_fw = (
    panel[(panel['year'] >= 2015) & (panel['year'] <= 2024)]
    .groupby('iso3')['renewable_freshwater_percap']
    .mean()
    .dropna()
    .sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Most water-scarce
recent_fw.head(20).plot.barh(ax=axes[0], color='#d62728')
axes[0].set_xlabel('Avg renewable freshwater per capita (m3/yr, 2015-2024)')
axes[0].set_title('20 most water-scarce countries')

# Most water-abundant
recent_fw.tail(20).sort_values(ascending=True).plot.barh(ax=axes[1], color='#2ca02c')
axes[1].set_xlabel('Avg renewable freshwater per capita (m3/yr, 2015-2024)')
axes[1].set_title('20 most water-abundant countries')

plt.tight_layout()
plt.show()

## 8. H7 — Groundwater Anomaly (GRACE)

Note: GRACE data requires running `load_grace()` on the downloaded netCDF4 file and adding to the panel. This cell will be enabled once the GRACE pipeline step is wired into assembly.

In [ ]:
if 'grace_lwe_anomaly_cm' in panel.columns:
    grace_trend = panel.groupby('year')['grace_lwe_anomaly_cm'].mean().dropna()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(grace_trend.index, grace_trend.values, color='steelblue')
    ax.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax.fill_between(grace_trend.index, grace_trend.values, 0,
                    where=grace_trend.values < 0, color='red', alpha=0.3, label='Deficit')
    ax.fill_between(grace_trend.index, grace_trend.values, 0,
                    where=grace_trend.values > 0, color='blue', alpha=0.3, label='Surplus')
    ax.set_title('Global mean groundwater storage anomaly (GRACE, cm LWE)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('GRACE data not yet in panel. Run grace ingest step in assembly pipeline.')

## 9. Key EDA Findings for Phase 3

In [ ]:
print('=== COVERAGE SUMMARY FOR PHASE 3 ===')
print()

hypotheses = [
    ('H1', 'Freshwater -> GDP', 'log_freshwater', 'gdp_pc_ppp'),
    ('H2', 'Freshwater -> Fragility', 'log_freshwater', 'fsi_score'),
    ('H3', 'Freshwater -> Conflict', 'log_freshwater', 'ucdp_conflict_binary'),
    ('H4', 'Freshwater -> Life exp', 'log_freshwater', 'life_expectancy'),
    ('H5', 'Freshwater -> Migration', 'log_freshwater', 'refugee_outflow'),
    ('H6', 'Freshwater -> Inequality', 'log_freshwater', 'gini'),
]

for h, label, x, y in hypotheses:
    n = recent[[x, y]].dropna().__len__()
    r = recent[[x, y]].dropna().corr().iloc[0, 1]
    countries = recent[[x, y, 'iso3']].dropna()['iso3'].nunique()
    print(f'{h}: {label}')
    print(f'   n={n:,} obs, {countries} countries, Spearman r={r:+.3f}')
    print()